<a href="https://colab.research.google.com/github/ehurtos/MLOPS/blob/main/Banka_to_CSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd

def extract_transactions(text):
   """
   Extracts transaction data from text and creates a DataFrame.
   """
   pattern = r"""
       ^(?P<Date>\d{2}Jul)?\s*
       (?P<Description>(?:(?![\d,.]+$).)+?)
       \s+
       (?P<Amount>[\d,.]+)
   """

   data = []
   last_date = None

   for line in text.split('\n'):
       try:
           matches = re.finditer(pattern, line.strip(), re.VERBOSE | re.MULTILINE)
           for match in matches:
               date = match.group('Date')
               if date is None:
                   date = last_date
               else:
                   last_date = date

               if date:
                   transaction = {
                       'Date': date,
                       'Description': match.group('Description').strip(),
                       'Amount': match.group('Amount')
                   }
                   data.append(transaction)
       except Exception as e:
           print(f"Error processing line: {line}\nError: {e}")

   return pd.DataFrame(data)

def clean_transactions(df):
   """
   Cleans transaction DataFrame.
   """
   try:
       df['Amount'] = df['Amount'].str.replace('$', '').str.replace(',', '').astype(float)
       df['Date'] = pd.to_datetime(df['Date'] + '2024', format='%d%b%Y')
       df = df[df['Amount'].notna()]
       return df.reset_index(drop=True)
   except Exception as e:
       print(f"Error cleaning DataFrame: {e}")
       return df

def process_bank_statement(text, output_file='bank.csv'):
   """
   Process bank statement and save to CSV.
   """
   try:
       df = extract_transactions(text)
       df = clean_transactions(df)
       df.to_csv(output_file, index=False)
       return df
   except Exception as e:
       print(f"Error processing bank statement: {e}")
       return None

text = """
Page 1:
RBBDA30000_6858598E D 00059 00738
EDSIGNACORPORATION
16INDIANARROWROAD
BARRIEON L4M5G3ROYALBANKOFCANADA
P.O.BOX4047TERMINALA
TORONTOON M5W1L5
1of2BusinessAccountStatement
June28,2024toJuly30,2024
AccountSummaryforthisPeriod
RBCFlexChoiceBusinessTMaccountpackage
RoyalBankofCanada
88QUEENSQUAREW-MAINFLR,TORONTO,ON M5J0B8
OpeningbalanceonJune28,2024 $162.52
Totaldeposits&credits(2) +20,776.40
Totalcheques&debits(16) -20,881.02
ClosingbalanceonJuly30,2024 = $57.90Accountnumber: 00059 100-114-8
Howtoreachus:
PleasecontactyourRBCBankingrepresentativeorcall
1-800-Royal®2-0
(1-800-769-2520)
www.rbcroyalbank.com/business
AccountActivityDetails
Date Description Cheques&Debits($) Deposits&Credits($) Balance($)
Openingbalance 162.52
02Jul Interacpurchase-5005B001 WEBERS 35.90
ContactlessInteracpurchase-7725B002
MTCSHURONIAHI 2.00
ContactlessInteracpurchase-8976B002
MTCSHURONIAHI 6.00
ContactlessInteracpurchase-0125B002
BINGHAM'SVARIE 7.35
BillPayment PAY-FILEFEES 2.00 109.27
Monthlyfee 7.00
Electronictransactionfee 12 Drs@ 0.75
2 Crs@ 0.75 10.50 91.77
05Jul Activityfee 25.00
Loaninterest NO.60993228001 11.68 55.09
12Jul AccountPayablePmt Procom 19,526.40
Page 2:
BusinessAccountStatement
2of2June28,2024toJuly30,2024
Accountnumber: 00059 100-114-8
AccountActivityDetails-continued
Date Description Cheques&Debits($) Deposits&Credits($) Balance($)
12Jul12Jul ATMwithdrawal-SJ722957 200.00
ContactlessInteracpurchase-1177B002 WEBERS 24.35
Onlinetransfersent-9027 Osobnyucet 10,000.00 9,357.14
LOANPAYMENT 250.00 9,107.14
15Jul OnlineBankingtransfer-8413 8,367.59 739.55
16Jul COMMERCIALTAXES EMPTX5846080 1,926.00 -1186.45
LOANCREDIT 1,250.00 63.55
26Jul ContactlessInteracpurchase-9051B002
SIXMILELAKEM 5.65 57.90
Closingbalance 57.90
AccountFees: $42.50
"""

if __name__ == "__main__":
   df = process_bank_statement(text)
   if df is not None:
       print(df)

         Date                                 Description    Amount
0  2024-07-02             Interacpurchase-5005B001 WEBERS     35.90
1  2024-07-02                               MTCSHURONIAHI      2.00
2  2024-07-02                               MTCSHURONIAHI      6.00
3  2024-07-02                              BINGHAM'SVARIE      7.35
4  2024-07-02                    BillPayment PAY-FILEFEES      2.00
5  2024-07-02                                  Monthlyfee      7.00
6  2024-07-02                    Electronictransactionfee     12.00
7  2024-07-02                                      2 Crs@      0.75
8  2024-07-05                                 Activityfee     25.00
9  2024-07-05                 Loaninterest NO.60993228001     11.68
10 2024-07-12                    AccountPayablePmt Procom  19526.40
11 2024-07-12                                        Page      2.00
12 2024-07-12                              Accountnumber:     59.00
13 2024-07-12                12Jul ATMwithdrawal